# Week 1 Day 5 Lab 2 - Harness Engineering

## Deep Agents works the queue on its own

Deep Agents is LangChain's ready-made harness. It's built on `create_agent`, the same function we used on Day 2, with middleware added for file tools, subagents, a to-do list and summarizing a long conversation. It runs on any model LangChain supports, so we can use OpenRouter.

This is the same job as Lab 1: a folder with five claims from the queue, the policy wording and the claims manual, and the instruction to decide every claim. We don't write any tools.

## 4 parts:

PART 1: The claims desk

PART 2: Setting up the harness

PART 3: Working the queue

PART 4: How it did

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls DeepSeek V4.1 Flash through OpenRouter, and a run on five claims costs about 3 cents.
            If you'd rather not spend anything, there's a free version of this lab in <a href="week1day5_lab2free.ipynb">week1day5_lab2free.ipynb</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import shutil
from pathlib import Path

import pandas as pd
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import TodoListMiddleware
from langchain_openai import ChatOpenAI

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

OpenRouter API key found


## Part 1: The claims desk

The claims desk is the same as in Lab 1. We use the five claims marked for the tiny version of the queue, which include one that needs the claims manual.

In [3]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [4]:
claims, labels = get_sample_data(tiny=True)

for claim in claims:
    print(claim["claim_id"], claim["summary"])

CLM-1007 Phone dropped in the hotel shower
CLM-1015 Flew home from Paris on arrival after leaving a child at home
CLM-1017 Backpack snatched by a scooter rider in Naples
CLM-1020 Prescription sunglasses stolen by a monkey at Uluwatu Temple
CLM-1021 Tote bag snatched from her shoulder in Buenos Aires


In [5]:
DESK = Path("lab2_desk")


def make_desk(claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    shutil.rmtree(DESK, ignore_errors=True)
    for folder in ["claims", "policies", "customers", "decisions"]:
        (DESK / folder).mkdir(parents=True)
    for claim in claims:
        (DESK / "claims" / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding="utf-8")
    for policy in load("policies.json"):
        (DESK / "policies" / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    for customer in load("customers.json"):
        (DESK / "customers" / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding="utf-8")
    for name in ["policy_wording.md", "claims_manual.md", "company.json"]:
        shutil.copy(DATA / name, DESK / name)


make_desk(claims)
sorted(path.name for path in DESK.iterdir())

['claims',
 'claims_manual.md',
 'company.json',
 'customers',
 'decisions',
 'policies',
 'policy_wording.md']

## Part 2: Setting up the harness

The instructions are the same as in Lab 1, apart from calling the task list a to-do list, which is Deep Agents' name for it.

In [6]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a to-do list with one item for each claim, and give each claim to the claim-handler subagent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Deep Agents</h2>
            <span style="color:#118AB2;"><code>create_deep_agent</code> gives the agent file tools (<code>ls</code>, <code>read_file</code>, <code>write_file</code>, <code>edit_file</code>, <code>glob</code>, <code>grep</code>) and <code>task</code>, which hands work to a subagent.<br/><br/>
            The to-do list is an extra since version 0.7, so we add it with LangChain's <code>TodoListMiddleware</code>, which gives the agent a <code>write_todos</code> tool.<br/><br/>
            <code>FilesystemBackend</code> points the file tools at the claims desk. With <code>virtual_mode=True</code> the agent sees the desk as <code>/</code> and can't reach anything outside it.
            </span>
        </td>
    </tr>
</table>

In [7]:
llm = ChatOpenAI(
    model="deepseek/deepseek-v4.1-flash",
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
    reasoning_effort="low",
    timeout=120,
    max_retries=5,
)

In [8]:
claim_handler = {
    "name": "claim-handler",
    "description": "Decides one claim and writes its decision file. Give it the claim id.",
    "system_prompt": HANDLER_INSTRUCTIONS,
}

agent = create_deep_agent(
    model=llm,
    system_prompt=INSTRUCTIONS,
    subagents=[claim_handler],
    middleware=[TodoListMiddleware()],
    backend=FilesystemBackend(root_dir=DESK, virtual_mode=True),
)

## Part 3: Working the queue

As the agent works we print each step and count the model calls and the cost. The stream gives us an update after each step of the agent, and the updates from the `model` step hold the model's replies. The subagent's steps are indented.

In [9]:
def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call["args"]
    if "file_path" in args or "path" in args:
        return f"{tool_call['name']} {args.get('file_path') or args.get('path')}"
    if tool_call["name"] == "task":
        return f"task: {args.get('description', '').splitlines()[0][:80]}"
    if tool_call["name"] == "write_todos":
        return "write_todos: " + ", ".join(f"{todo['content'][:30]} ({todo['status']})" for todo in args.get("todos", []))
    return tool_call["name"]

In [10]:
cost = 0
calls = 0

for namespace, update in agent.stream(
    {"messages": [{"role": "user", "content": "Please work the queue."}]},
    stream_mode="updates",
    subgraphs=True,
    config={"recursion_limit": 200},
):
    if "model" in update:
        indent = "      " if namespace else ""
        for message in update["model"]["messages"]:
            calls += 1
            cost += message.response_metadata["token_usage"]["cost"]
            for tool_call in message.tool_calls:
                print(indent + describe(tool_call))
            if not namespace and message.text.strip():
                print(message.text.strip())

print(f"\nFinished after {calls} model calls for ${cost:.4f}")

ls /
ls /claims
ls /decisions
read_file /claims_manual.md
read_file /company.json
read_file /policy_wording.md
read_file /claims_manual.md
read_file /policy_wording.md
ls /policies
ls /customers
read_file /claims/CLM-1007.json
read_file /claims/CLM-1015.json
read_file /claims/CLM-1017.json
read_file /claims/CLM-1020.json
read_file /claims/CLM-1021.json
read_file /policies/POL-2007.json
read_file /policies/POL-2015.json
read_file /policies/POL-2017.json
read_file /policies/POL-2020.json
read_file /policies/POL-2021.json
read_file /customers/CUS-3007.json
read_file /customers/CUS-3015.json
read_file /customers/CUS-3017.json
read_file /customers/CUS-3020.json
read_file /customers/CUS-3021.json
write_todos: Decide CLM-1007 (phone water d (in_progress), Decide CLM-1015 (trip interrup (in_progress), Decide CLM-1017 (Naples backpa (in_progress), Decide CLM-1020 (Bali monkey,  (in_progress), Decide CLM-1021 (Buenos Aires  (in_progress), Write handover.md at top of fo (pending)
task: Decide cla

## Part 4: How it did

We score the decision files with the same test as in Lab 1.

In [11]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer.get("decision") == label["decision"] and abs(float(answer.get("payout") or 0) - label["payout"]) <= 1


rows = []
for claim_id, label in labels.items():
    path = DESK / "decisions" / f"{claim_id}.json"
    answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
    rows.append({
        "claim": claim_id,
        "decision": answer.get("decision"),
        "payout": answer.get("payout"),
        "handler's decision": label["decision"],
        "handler's payout": label["payout"],
        "right": is_right(answer, label),
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results

5 of 5 right


,claim,decision,payout,handler's decision,handler's payout,right
0,CLM-1007,approve,50.0,approve,50.0,True
1,CLM-1015,decline,0.0,decline,0.0,True
2,CLM-1017,refer,0.0,refer,0.0,True
3,CLM-1020,approve,240.0,approve,240.0,True
4,CLM-1021,approve,825.0,approve,825.0,True


And here's the note it left for the handler.

In [12]:
display(Markdown((DESK / "handover.md").read_text(encoding="utf-8")))

# Handover — claim queue (all five decided)

Wording version 2026.1, claims manual 2026.2. All decisions were made by "claim-handler" on the claim's filing/decision date. **No payout exceeded $1,000, so none needs a second handler's sign-off.** Decision files are in `/decisions/`.

## Summary

| Claim | Customer | Type | Decision | Payout | Key basis |
|---|---|---|---|---|---|
| CLM-1007 | Greg Kowalski | Baggage (accidental damage) | Approve | $50.00 | §5; Economy per-item $150, less $100 deductible |
| CLM-1015 | Kate Callister | Trip interruption | Decline | $0.00 | §2; reason not on the covered list |
| CLM-1017 | Emily Nguyen | Baggage (theft) | Refer | $0.00 (pot. $440) | §5; police report missing |
| CLM-1020 | Nadia Rahman | Baggage (wild animal) | Approve | $240.00 | §5 + manual §7 wild-animal note; no deductible on First |
| CLM-1021 | Camille Fontaine | Baggage (theft) | Approve | $825.00 | §5 + §8 Electronics Plus; less $50 deductible |

## Notes the handler should know

- **CLM-1007 (approve $50):** The headline number ($1,099 replacement phone) is far above what the Economy plan pays. The per-item limit is $150 and the $100 deductible comes off, so this is a deliberately small payout, not an error. No sign-off needed.
- **CLM-1015 (decline):** A sympathetic story — the family accidentally flew to Paris without their 8-year-old — but §2 lists only five covered reasons and closes with "we do not pay for cancellation or interruption for any other reason." The documents are consistent, so this is a clean decline, not a referral. The letter should be kind; nothing here suggests dishonesty.
- **CLM-1017 (refer):** Genuine theft-from-the-person (a snatch, so not "unattended"), but the Carabinieri report is not on file. A 30-day document request is recorded in the decision. **Also chase proof of value for the sunglasses ($130)** — the receipts on file cover only the tablet, jacket and daypack. If the report arrives, the payout would be **$440** (items capped at the $150 Economy per-item limit, less the $100 deductible).
- **CLM-1020 (approve $240):** Loss by a wild animal (a macaque took the prescription sunglasses at Uluwatu Temple). Manual §7 treats wild-animal loss as accidental loss, **not theft**, so no police report is required and the licensed guide's statement plus the $240 receipt are sufficient. First plan has no deductible.
- **CLM-1021 (approve $825):** Electronics Plus raises the per-item limit to $1,200 only for the items the wording defines as electronics. Only the **e-reader ($180)** qualifies; the **headphones ($429)** and **game console ($399)** are not on that list and stay at the Premium per-item limit of $300, as does the tote bag. The customer may query why the add-on did not lift all three electronics — the letter should show the line-by-line calculation and explain the definition.

## Housekeeping

- No claim was the customer's third in 12 months (all customers have no previous claims), so no SIU referral for repeat claims.
- No story/document conflicts were found that would require the Special Investigations Unit.
- All incidents fall inside the covered trip dates and all claims were filed within 90 days.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Build or buy</h2>
            <span style="color:#118AB2;">Deep Agents is open source and runs on any model, so you can pick a cheap one and read exactly what the harness does. The Claude Agent SDK gives you Claude Code's harness, tuned for Claude models.<br/><br/>
            Next we build the same pieces into UnderCover ourselves: a to-do list, tools to read the manual, and a subagent for each claim.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Look back through the steps and find where the agent used its to-do list and where it handed work to the subagent.<br/><br/>
            Then try other models and compare accuracy, cost and time: <code>openai/gpt-6-luna</code>, or <code>anthropic/claude-sonnet-5.5</code> to compare with Lab 1 on the same model, which costs more.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>